# IPL Matches — Data Exploration & Cleaning Pipeline Run

This notebook is **not** where the cleaning logic lives — every cleaning
operation is written as its own reviewable `.sql` file in `../sql/`.

This notebook has two jobs:
1. **Explore** the raw dataset to show *why* each cleaning step in `../sql/` is needed.
2. **Run** the 8 `.sql` files in order against an in-memory SQLite database (exactly as a grader would with SQLime or the sqlite3 CLI) and confirm the final `output/matches_clean.csv` matches what those scripts produce.

In [1]:
import sqlite3
import pandas as pd

RAW_PATH = "../data/raw/matches.csv"
raw = pd.read_csv(RAW_PATH)
raw.shape

(1212, 28)

## 1. Explore the raw data

In [2]:
raw.head()

,match_id,season_id,balls_per_over,city,match_date,event_name,match_number,gender,match_type,format,...,match_winner,win_by_runs,win_by_wickets,result,player_of_match,team1_id,team2_id,toss_winner_id,match_winner_id,player_of_match_id
0,335982,2008,6,Bangalore,2008-04-18,Indian Premier League,1.0,male,T20,T20,...,Kolkata Knight Riders,140.0,NaN,win,BB McCullum,1,6,1,6,46.0
1,1082591,2017,6,Hyderabad,2017-04-05,Indian Premier League,1.0,male,T20,T20,...,Sunrisers Hyderabad,35.0,NaN,win,Yuvraj Singh,2,1,1,2,15.0
2,1082592,2017,6,Pune,2017-04-06,Indian Premier League,2.0,male,T20,T20,...,Rising Pune Supergiant,NaN,7.0,win,SPD Smith,4,3,4,4,36.0
3,1082593,2017,6,Rajkot,2017-04-07,Indian Premier League,3.0,male,T20,T20,...,Kolkata Knight Riders,NaN,10.0,win,CA Lynn,5,6,6,6,57.0
4,1082594,2017,6,Indore,2017-04-08,Indian Premier League,4.0,male,T20,T20,...,Punjab Kings,NaN,6.0,win,GJ Maxwell,494,4,494,494,71.0


In [3]:
# Missing values per column
raw.isna().sum().sort_values(ascending=False)

win_by_runs           666
win_by_wickets        571
match_number           70
city                   51
player_of_match_id      9
player_of_match         9
event_name              0
gender                  0
balls_per_over          0
match_date              0
season_id               0
match_id                0
season                  0
overs                   0
format                  0
match_type              0
team1                   0
team_type               0
toss_winner             0
venue                   0
match_winner            0
toss_decision           0
team2                   0
result                  0
team1_id                0
team2_id                0
toss_winner_id          0
match_winner_id         0
dtype: int64

In [4]:
# Venue names: raw data has far more distinct spellings than real venues
print("distinct raw venue strings:", raw['venue'].nunique())
sorted(raw['venue'].unique())[:15]

distinct raw venue strings: 59


['Arun Jaitley Stadium',
 'Arun Jaitley Stadium, Delhi',
 'Barabati Stadium',
 'Barsapara Cricket Stadium, Guwahati',
 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow',
 'Brabourne Stadium',
 'Brabourne Stadium, Mumbai',
 'Buffalo Park',
 'De Beers Diamond Oval',
 'Dr DY Patil Sports Academy',
 'Dr DY Patil Sports Academy, Mumbai',
 'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium',
 'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam',
 'Dubai International Cricket Stadium',
 'Eden Gardens']

In [5]:
# Team names across team1/team2 — checked for abbreviations/case variants
teams = sorted(set(raw['team1'].unique()) | set(raw['team2'].unique()))
print("distinct team names:", len(teams))
teams

distinct team names: 14


['Chennai Super Kings',
 'Delhi Capitals',
 'Gujarat Lions',
 'Gujarat Titans',
 'Kochi Tuskers Kerala',
 'Kolkata Knight Riders',
 'Lucknow Super Giants',
 'Mumbai Indians',
 'Pune Warriors',
 'Punjab Kings',
 'Rajasthan Royals',
 'Rising Pune Supergiant',
 'Royal Challengers Bangalore',
 'Sunrisers Hyderabad']

In [6]:
# Rows where an abandoned ('no result') match still has a match_winner set
raw[raw['result'] == 'no result'][['match_id', 'result', 'match_winner', 'player_of_match']]

,match_id,result,match_winner,player_of_match
168,1178424,no result,Rajasthan Royals,NaN
418,1359519,no result,Chennai Super Kings,NaN
767,501265,no result,Pune Warriors,NaN
1012,829763,no result,Rajasthan Royals,NaN
1037,829813,no result,Delhi Capitals,NaN
1138,1473481,no result,Kolkata Knight Riders,NaN
1149,1473492,no result,Sunrisers Hyderabad,NaN
1152,1473495,no result,Delhi Capitals,NaN
1180,1527685,no result,Punjab Kings,NaN


In [7]:
# Season labels — note the real '2020/21' UAE-season label
sorted(raw['season'].astype(str).unique())

['2008',
 '2009',
 '2010',
 '2011',
 '2012',
 '2013',
 '2014',
 '2015',
 '2016',
 '2017',
 '2018',
 '2019',
 '2020/21',
 '2021',
 '2022',
 '2023',
 '2024',
 '2025',
 '2026']

## 2. Run the SQL cleaning pipeline (`../sql/01` → `../sql/08`)

In [8]:
SQL_FILES = [
    "01_nulls_and_blanks.sql",
    "02_merge_categories.sql",
    "03_venue_names.sql",
    "04_dedupe_venues.sql",
    "05_city_fallback.sql",
    "06_season_year.sql",
    "07_win_definition.sql",
    "08_matches_clean.sql",
]

con = sqlite3.connect(":memory:")
raw.to_sql("matches", con, index=False)

cur = con.cursor()
for fname in SQL_FILES:
    with open(f"../sql/{fname}") as f:
        script = f.read()
    cur.executescript(script)
    con.commit()
    print(f"ran {fname}")

ran 01_nulls_and_blanks.sql
ran 02_merge_categories.sql
ran 03_venue_names.sql
ran 04_dedupe_venues.sql
ran 05_city_fallback.sql
ran 06_season_year.sql
ran 07_win_definition.sql
ran 08_matches_clean.sql


## 3. Inspect the cleaned result

In [9]:
clean = pd.read_sql("SELECT * FROM matches_clean", con)
clean.shape

(1212, 29)

In [10]:
clean.isna().sum().sort_values(ascending=False)

win_by_runs           666
win_by_wickets        571
match_number           70
match_winner            9
team1_won               9
player_of_match         9
player_of_match_id      9
match_winner_id         9
event_name              0
match_date              0
match_id                0
season_label            0
season_year             0
toss_decision           0
toss_winner             0
team2                   0
team1                   0
city                    0
venue                   0
result                  0
balls_per_over          0
format                  0
overs                   0
match_type              0
gender                  0
team1_id                0
team_type               0
toss_winner_id          0
team2_id                0
dtype: int64

In [11]:
print("distinct venues after cleaning:", clean['venue'].nunique())
print("distinct teams after cleaning:", pd.concat([clean['team1'], clean['team2']]).nunique())
print("duplicate match_id rows:", clean['match_id'].duplicated().sum())
clean[['result']].value_counts()

distinct venues after cleaning: 36
distinct teams after cleaning: 14
duplicate match_id rows: 0


result   
win          1187
tie            16
no result       9
Name: count, dtype: int64

In [12]:
# Quick sanity plot: matches per season_year
clean.groupby('season_year')['match_id'].count()

season_year
2008    58
2009    57
2010    60
2011    73
2012    74
2013    76
2014    60
2015    59
2016    60
2017    59
2018    60
2019    60
2020    60
2021    60
2022    74
2023    74
2024    71
2025    74
2026    43
Name: match_id, dtype: int64

## 4. Save the final cleaned dataset

Writes `../output/matches_clean.csv` — the same file produced by running
`08_matches_clean.sql` in SQLime/sqlite3 and exporting `matches_clean`.

In [13]:
clean.to_csv("../output/matches_clean.csv", index=False)
print("saved ../output/matches_clean.csv")

saved ../output/matches_clean.csv
